In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)


In [ ]:
FT2026_SRC = pd.read_csv(r"S:\15.09.26\30919829_NEWCNEFINTRANSRPT.CSV", on_bad_lines='skip',dtype='str')
FT2025_SRC = pd.read_csv(r"S:\2026\01-Jan-26\01.01.26\28687976_NEWCNEFINTRANSRPT.CSV", on_bad_lines='skip',dtype='str')
BEINDATA_SRC = pd.read_csv(r"S:\15.09.26\15092026_BEINDATANEWRPT.csv", encoding='cp1256',dtype='str')


In [87]:
ft2026 = FT2026_SRC.copy()
ft2025 = FT2025_SRC.copy()
beindata = BEINDATA_SRC.copy()

showrooms = [ "Maadi showroom","Mohandeseen Showroom"]
dth_types = ['beIN Quartar Installment', 'CNE Subscriber', 'MCE staff (CNE staff)',
                'BeIN sports CC', 'beIN Bi Installment', 'Corporate Subscriber', 'Temp',
                'Bein NC', 'Bulk DTH customer', 'beIN Installment Sub', 'Charge Back']

plan_filter = (
            beindata["PLAN"].str.contains(
                "prem",
                case=False,
                na=False
            )
            |
            beindata["PLAN"].str.contains(
                "ulti",
                case=False,
                na=False
            )
            |
            beindata["PLAN"].str.contains(
                "toget",
                case=False,
                na=False
            )
        )

2025

In [89]:
ft2025['Created Date'] = pd.to_datetime(ft2025['Created Date'], dayfirst=True)

ft2025 = ft2025.loc[ft2025['Created Date'].between(pd.to_datetime('2025-08-01'),pd.to_datetime('2025-08-31'))]

ft2025 = ft2025.loc[ft2025['Collecting Entity'].isin(showrooms)]
ft2025 = ft2025.loc[ft2025['Subscriber Type'].isin(dth_types)]
ft2025 = ft2025.loc[(ft2025['Doc Type'] =='Payment') & (ft2025['Doc Status'] =='Posted')]


ft2025 = ft2025.drop_duplicates(subset=['Subscriber Nr'])
print(f'2025 walk in subs: {ft2025.shape[0]}')


2025 walk in subs: 1736


2026

In [90]:
ft2026['Created Date'] = pd.to_datetime(ft2026['Created Date'], dayfirst=True)
ft2026 = ft2026.loc[ft2026['Created Date'].between(pd.to_datetime('2026-07-01'),pd.to_datetime('2026-09-30'))]
ft2026 = ft2026.loc[(ft2026['Doc Type'] =='Payment') & (ft2026['Doc Status'] =='Posted')]


In [91]:
beindata = beindata.loc[plan_filter]
beindata = beindata.sort_values(['Customer Number' , 'STATUS'], ascending=[True,True])
beindata = beindata.drop_duplicates(subset=['Customer Number'])
beindata = beindata.dropna(subset=['STATUS'])

In [92]:
display(ft2025.columns)
display(beindata.columns)

Index(['Subscriber Nr', 'Doc Type', 'Ftnr', 'Created Date', 'Created Time',
       'Doc Status', 'Period From', 'Period To', 'Bank Date', 'Amount',
       'User Name', 'User Fullname', 'Payment Ref No', 'Event Description',
       'Payment Batch No', 'Batch Approval No', 'Default Entity Type',
       'Collecting Entity', 'Pay Mode', 'Jv Type', 'Book Number', 'Smartcard',
       'Bill Period', 'Bill Cycle', 'Invoice Type', 'Plan Name',
       'Contract Number', 'Channel Provider', 'Subscriber Type',
       'Subscriber Entity', 'Last Four Digits Of Card', 'Payment Flag'],
      dtype='object')

Index(['Customer Number', 'Customer Type', 'ENTITY', 'Contract Number',
       'Start Date', 'End Date', 'PLAN', 'STATUS', 'DECODER',
       'Item Description STB', 'Smart Card', 'Item Description SC',
       'Next Billing Date', ' Billing Cycle', 'PPV Balance',
       'Customer Balance', 'Outstanding Balance'],
      dtype='object')

In [ ]:
ft2025 = ft2025.merge(right=beindata[['Customer Number','PLAN','STATUS']], left_on='Subscriber Nr', right_on='Customer Number', how='inner')
ft2025.shape[0]

In [ ]:
ft2025.head(3)

In [ ]:
ft2025['is_in_2026'] = False

ft2025.loc[ft2025['Subscriber Nr'].isin(ft2026['Subscriber Nr']),'is_in_2026'] = True
# ft2025_not_in_2026 = ft2025.loc[~ft2025['Subscriber Nr'].isin(ft2026['Subscriber Nr'])]




In [85]:
ft2025.head(2)

,Subscriber Nr,Doc Type,Ftnr,Created Date,Created Time,Doc Status,Period From,Period To,Bank Date,Amount,User Name,User Fullname,Payment Ref No,Event Description,Payment Batch No,Batch Approval No,Default Entity Type,Collecting Entity,Pay Mode,Jv Type,Book Number,Smartcard,Bill Period,Bill Cycle,Invoice Type,Plan Name,Contract Number,Channel Provider,Subscriber Type,Subscriber Entity,Last Four Digits Of Card,Payment Flag,Customer Number,PLAN,STATUS,is_in_2026
0,19199142,Payment,CNEPMT_2071147,2025-08-01,09:23:06 AM,Posted,NaN,NaN,NaN,5340,MOKARAM,Mohamed Karam Mohamed Morad,1702045,DC 134223,BM MOAUG 01/25 CC,NaN,CNE Head Office,Mohandeseen Showroom,Credit Card,NaN,NaN,10732785265,NaN,NaN,NaN,NaN,NaN,beIN,CNE Subscriber,CNE Head office,4964,Hardware,19199142,PREMIUM 09.24,DIS,True
1,19199172,Payment,CNEPMT_2071192,2025-08-01,02:13:08 PM,Posted,NaN,NaN,NaN,5340,HMONIER,Hazem Mohamed Monier Mohamed,NaN,DC 359642,AAIB AUG 01/25 CC,NaN,CNE Head Office,Maadi showroom,Credit Card,NaN,NaN,10732866222,NaN,NaN,NaN,NaN,NaN,beIN,BeIN sports CC,Maadi showroom,6786,Hardware,19199172,ULTIMATE 09.24,Active,True


In [86]:
ft2025.to_csv("showrooms_comparison.csv", index= False)